In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║    DIADORA COMPETITIVE ANALYSIS SCRAPER - VERSIONE MULTI-BRAND v4.0         ║
║                                                                              ║
║    Brand analizzati:                                                         ║
║      - Diadora  (diadora.com/it/it/)                                        ║
║      - Nike     (nike.com/it/)                                               ║
║      - Adidas   (adidas.it/)                                                 ║
║      - Puma     (eu.puma.com/it/)                                            ║
║      - New Balance (newbalance.com/it/)                                      ║
║      - Asics    (asics.com/it/)                                              ║
║                                                                              ║
║    Dati raccolti per brand:                                                  ║
║      - Nome modello, prezzo, categoria, colori, taglie disponibili          ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import time
import re
import json
from datetime import datetime
from typing import List, Dict, Optional, Tuple
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.series import DataPoint


# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAZIONE BRAND E SELETTORI
# ─────────────────────────────────────────────────────────────────────────────

BRAND_CONFIGS: Dict[str, Dict] = {

    "Diadora": {
        "base_url": "https://www.diadora.com/it/it/",
        "shoe_url": "https://www.diadora.com/it/it/calzature/",
        "cookie_selectors": [
            (By.ID, "onetrust-accept-btn-handler"),
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
            (By.XPATH, "//button[contains(text(), 'Accept')]"),
        ],
        "product_container": [
            ("div", r"product-grid__item"),
            ("div", r"product-tile"),
            ("li",  r"product-item"),
        ],
        "name_selectors": [
            ("class", r"product-tile__name"),
            ("class", r"product-name"),
        ],
        "price_selectors": [
            ("class", r"product-price__sale"),
            ("class", r"price"),
        ],
        "color_selectors": [
            ("class", r"color-swatches"),
            ("class", r"swatch"),
        ],
        "size_selectors": [
            ("class", r"size-selector"),
            ("class", r"sizes"),
        ],
        "category_url_map": {
            "Sneakers": "https://www.diadora.com/it/it/calzature/sneakers/",
            "Running":  "https://www.diadora.com/it/it/calzature/running/",
            "Calcio":   "https://www.diadora.com/it/it/calzature/calcio/",
            "Sandali":  "https://www.diadora.com/it/it/calzature/sandali/",
            "Stivali":  "https://www.diadora.com/it/it/calzature/stivali/",
        },
        "pagination_selector": (By.XPATH, "//a[@class='pagination__next']"),
    },

    "Nike": {
        "base_url": "https://www.nike.com/it/",
        "shoe_url": "https://www.nike.com/it/w/scarpe-y7ok",
        "cookie_selectors": [
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
            (By.XPATH, "//button[@id='hf-cookie-accept-btn']"),
        ],
        "product_container": [
            ("div",     r"product-card"),
            ("div",     r"product-card__body"),
        ],
        "name_selectors": [
            ("class", r"product-card__title"),
            ("class", r"product-card__subtitle"),
        ],
        "price_selectors": [
            ("class", r"product-price"),
            ("class", r"css-.*-ProductPrice"),
        ],
        "color_selectors": [
            ("class", r"product-card__product-count"),
        ],
        "size_selectors": [],
        "category_url_map": {
            "Sneakers":  "https://www.nike.com/it/w/sneakers-5e1x6",
            "Running":   "https://www.nike.com/it/w/running-37v7j",
            "Calcio":    "https://www.nike.com/it/w/scarpe-da-calcio-1gdj0",
            "Sandali":   "https://www.nike.com/it/w/sandali-2po3y",
            "Training":  "https://www.nike.com/it/w/training-5e1x6",
        },
        "pagination_selector": (By.XPATH, "//button[@data-qa='load-more-button']"),
    },

    "Adidas": {
        "base_url": "https://www.adidas.it/",
        "shoe_url": "https://www.adidas.it/scarpe",
        "cookie_selectors": [
            (By.XPATH, "//button[contains(@data-auto-id, 'gdpr-consent-accept')]"),
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
        ],
        "product_container": [
            ("div",  r"product-card"),
            ("div",  r"\bgl-product-card\b"),
        ],
        "name_selectors": [
            ("class", r"product-card__title"),
            ("class", r"gl-product-card__name"),
        ],
        "price_selectors": [
            ("class", r"product-card__pricing"),
            ("class", r"gl-product-card__price"),
        ],
        "color_selectors": [
            ("class", r"product-card__color"),
        ],
        "size_selectors": [],
        "category_url_map": {
            "Sneakers":  "https://www.adidas.it/scarpe-lifestyle",
            "Running":   "https://www.adidas.it/scarpe-running",
            "Calcio":    "https://www.adidas.it/scarpe-calcio",
            "Sandali":   "https://www.adidas.it/sandali",
            "Training":  "https://www.adidas.it/scarpe-training",
            "Tennis":    "https://www.adidas.it/scarpe-tennis",
        },
        "pagination_selector": (By.XPATH, "//button[contains(@class, 'pagination__btn--next')]"),
    },

    "Puma": {
        "base_url": "https://eu.puma.com/it/it/",
        "shoe_url": "https://eu.puma.com/it/it/scarpe",
        "cookie_selectors": [
            (By.XPATH, "//button[contains(@class, 'accept')]"),
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
        ],
        "product_container": [
            ("div", r"product-tile"),
            ("article", r"product"),
        ],
        "name_selectors": [
            ("class", r"product-tile__name"),
            ("class", r"product-name"),
        ],
        "price_selectors": [
            ("class", r"product-price"),
            ("class", r"price"),
        ],
        "color_selectors": [
            ("class", r"product-tile__color"),
        ],
        "size_selectors": [],
        "category_url_map": {
            "Sneakers":  "https://eu.puma.com/it/it/sneakers",
            "Running":   "https://eu.puma.com/it/it/running",
            "Calcio":    "https://eu.puma.com/it/it/calcio/scarpe",
            "Sandali":   "https://eu.puma.com/it/it/sandali",
            "Training":  "https://eu.puma.com/it/it/training",
        },
        "pagination_selector": (By.XPATH, "//button[contains(@class, 'load-more')]"),
    },

    "New Balance": {
        "base_url": "https://www.newbalance.com/it/",
        "shoe_url": "https://www.newbalance.com/it/mens/footwear/",
        "cookie_selectors": [
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
            (By.ID, "onetrust-accept-btn-handler"),
        ],
        "product_container": [
            ("li",  r"product-tile"),
            ("div", r"product-tile"),
        ],
        "name_selectors": [
            ("class", r"product-tile__product-name"),
        ],
        "price_selectors": [
            ("class", r"product-tile-price"),
        ],
        "color_selectors": [
            ("class", r"product-tile-colors-count"),
        ],
        "size_selectors": [],
        "category_url_map": {
            "Sneakers": "https://www.newbalance.com/it/uomo/calzature/lifestyle/",
            "Running":  "https://www.newbalance.com/it/uomo/calzature/running/",
            "Training": "https://www.newbalance.com/it/uomo/calzature/training/",
            "Tennis":   "https://www.newbalance.com/it/uomo/calzature/tennis/",
        },
        "pagination_selector": (By.XPATH, "//button[contains(@class, 'pagination')]"),
    },

    "Asics": {
        "base_url": "https://www.asics.com/it/",
        "shoe_url": "https://www.asics.com/it/it-it/scarpe/",
        "cookie_selectors": [
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
            (By.ID, "onetrust-accept-btn-handler"),
        ],
        "product_container": [
            ("li",  r"product-grid-item"),
            ("div", r"product-tile"),
        ],
        "name_selectors": [
            ("class", r"product-tile__title"),
        ],
        "price_selectors": [
            ("class", r"price"),
            ("class", r"product-tile__price"),
        ],
        "color_selectors": [
            ("class", r"color-swatches"),
        ],
        "size_selectors": [],
        "category_url_map": {
            "Running":   "https://www.asics.com/it/it-it/scarpe/running/",
            "Tennis":    "https://www.asics.com/it/it-it/scarpe/tennis/",
            "Training":  "https://www.asics.com/it/it-it/scarpe/training/",
            "Sneakers":  "https://www.asics.com/it/it-it/scarpe/lifestyle/",
        },
        "pagination_selector": (By.XPATH, "//a[contains(@class, 'next-page')]"),
    },
}


# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFICATORE CATEGORIE
# ─────────────────────────────────────────────────────────────────────────────

CATEGORY_RULES: Dict[str, Dict] = {
    "Running": {
        "keywords": ["run", "running", "corsa", "jogging", "marathon", "gel", "fresh foam", "boost"],
        "subcategories": {
            "Strada":      ["road", "strada", "urban"],
            "Trail":       ["trail", "mountain", "off-road"],
            "Racing":      ["race", "racing", "carbon", "pro"],
            "Allenamento": ["training", "everyday", "daily"],
        }
    },
    "Calcio": {
        "keywords": ["calcio", "football", "soccer", "fg", "ag", "tf", "sg", "indoor", "futsal"],
        "subcategories": {
            "Terreno Naturale":  ["fg", "firm ground", "naturale"],
            "Sintetico":         ["ag", "artificial", "sintetico", "tf"],
            "Indoor":            ["indoor", "sala", "futsal", "in"],
        }
    },
    "Tennis": {
        "keywords": ["tennis", "court", "clay", "clay court", "grass", "hard court"],
        "subcategories": {
            "Terra Rossa": ["clay"],
            "Cemento":     ["hard", "all court", "allcourt"],
            "Erba":        ["grass"],
        }
    },
    "Sneakers": {
        "keywords": ["sneaker", "lifestyle", "casual", "retro", "classic", "street", "heritage"],
        "subcategories": {
            "Retro/Heritage": ["retro", "heritage", "classic", "og", "vintage", "original"],
            "Contemporanee":  ["lifestyle", "casual", "contemporary", "modern"],
            "Collab/Limited": ["collab", "limited", "special edition", "x ", " x "],
        }
    },
    "Training": {
        "keywords": ["training", "cross", "gym", "workout", "fitness", "functional"],
        "subcategories": {
            "Cross Training": ["cross", "crossfit"],
            "Gym":            ["gym", "fitness", "indoor"],
            "Functional":     ["functional", "performance"],
        }
    },
    "Sandali": {
        "keywords": ["sandal", "sandali", "slide", "flip", "ciabatta", "infradito"],
        "subcategories": {
            "Slides":       ["slide", "ciabatta"],
            "Flip Flop":    ["flip", "infradito"],
            "Sandali Bassi":["sandal", "sandali", "flat"],
            "Sandali Alti": ["wedge", "heel", "tacco"],
        }
    },
    "Stivali": {
        "keywords": ["stival", "boot", "ankle", "chelsea", "combat"],
        "subcategories": {
            "Chelsea":      ["chelsea"],
            "Ankle Boot":   ["ankle"],
            "Combat":       ["combat", "military"],
            "Stivali Alti": ["tall boot", "stivale alto"],
        }
    },
    "Scarpe con tacco": {
        "keywords": ["heel", "tacco", "pump", "stiletto", "wedge"],
        "subcategories": {
            "Stiletto":     ["stiletto"],
            "Pump":         ["pump"],
            "Wedge":        ["wedge"],
            "Kitten Heel":  ["kitten"],
        }
    },
}


def classify_product(name: str, url_category: str = "") -> Tuple[str, str]:
    """
    Classifica un prodotto in categoria e sottocategoria.
    Restituisce (categoria, sottocategoria).
    """
    name_lower = (name + " " + url_category).lower()

    for category, rules in CATEGORY_RULES.items():
        if any(kw in name_lower for kw in rules["keywords"]):
            subcategory = "Generale"
            for sub_name, sub_keywords in rules["subcategories"].items():
                if any(sk in name_lower for sk in sub_keywords):
                    subcategory = sub_name
                    break
            return category, subcategory

    return "Altro", "Altro"


# ─────────────────────────────────────────────────────────────────────────────
# SCRAPER BASE
# ─────────────────────────────────────────────────────────────────────────────

class BrandScraper:
    """Scraper generico configurabile per qualsiasi brand"""

    def __init__(self, brand_name: str, config: Dict, driver: webdriver.Chrome):
        self.brand_name = brand_name
        self.config = config
        self.driver = driver

    def handle_cookies(self):
        for by, selector in self.config["cookie_selectors"]:
            try:
                btn = WebDriverWait(self.driver, 4).until(
                    EC.element_to_be_clickable((by, selector))
                )
                btn.click()
                print(f"   ✅ [{self.brand_name}] Cookie accettati")
                time.sleep(1)
                return
            except:
                continue
        print(f"   ⚠️  [{self.brand_name}] Nessun cookie banner trovato")

    def scroll_to_load(self, max_scrolls: int = 12):
        """Scroll infinito per caricare tutti i prodotti"""
        last_height = self.driver.execute_script("return document.body.scrollHeight")
        for i in range(max_scrolls):
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2.5)

            # Prova click su "carica altri" se presente
            try:
                more_btn = WebDriverWait(self.driver, 2).until(
                    EC.element_to_be_clickable(self.config["pagination_selector"])
                )
                more_btn.click()
                time.sleep(2)
            except:
                pass

            new_height = self.driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

    def extract_text_by_class_pattern(self, container: BeautifulSoup, patterns: List) -> Optional[str]:
        """Cerca testo usando pattern regex sulle classi CSS"""
        for selector_type, pattern in patterns:
            if selector_type == "class":
                elem = container.find(class_=re.compile(pattern, re.I))
                if elem:
                    return elem.get_text(strip=True)
        return None

    def extract_price(self, container: BeautifulSoup) -> Optional[float]:
        """Estrae e pulisce un prezzo numerico"""
        text = self.extract_text_by_class_pattern(container, self.config["price_selectors"])
        if text:
            # Cerca pattern tipo "129,99" o "129.99"
            matches = re.findall(r'\d{1,4}[,\.]\d{2}', text)
            if matches:
                price_str = matches[0].replace(",", ".")
                try:
                    return float(price_str)
                except:
                    pass
            # Fallback: solo numero intero
            matches = re.findall(r'\d{2,4}', text)
            if matches:
                try:
                    return float(matches[0])
                except:
                    pass
        return None

    def extract_colors_count(self, container: BeautifulSoup) -> int:
        """Conta il numero di colori disponibili"""
        for selector_type, pattern in self.config["color_selectors"]:
            if selector_type == "class":
                elem = container.find(class_=re.compile(pattern, re.I))
                if elem:
                    # Cerca "N colori" nel testo
                    text = elem.get_text()
                    match = re.search(r'(\d+)', text)
                    if match:
                        return int(match.group(1))
                    # Conta i figli swatch
                    swatches = elem.find_all(class_=re.compile(r'swatch|color-option', re.I))
                    if swatches:
                        return len(swatches)
        return 1  # Default: almeno 1 colore

    def extract_sizes(self, container: BeautifulSoup) -> List[str]:
        """Estrae le taglie disponibili"""
        for selector_type, pattern in self.config.get("size_selectors", []):
            if selector_type == "class":
                elem = container.find(class_=re.compile(pattern, re.I))
                if elem:
                    sizes = elem.find_all(class_=re.compile(r'size|taglia', re.I))
                    return [s.get_text(strip=True) for s in sizes if s.get_text(strip=True)]
        return []

    def find_product_containers(self, soup: BeautifulSoup) -> List:
        """Trova tutti i container prodotto nella pagina"""
        for tag, pattern in self.config["product_container"]:
            containers = soup.find_all(tag, class_=re.compile(pattern, re.I))
            if len(containers) >= 3:
                print(f"   ✅ [{self.brand_name}] Trovati {len(containers)} prodotti con '{pattern}'")
                return containers
        print(f"   ⚠️  [{self.brand_name}] Container prodotti non trovato")
        return []

    def scrape_category(self, category: str, url: str, max_products: int = 100) -> List[Dict]:
        """Scrapa una singola categoria URL"""
        print(f"\n   📂 [{self.brand_name}] Categoria: {category}")
        print(f"      URL: {url}")

        try:
            self.driver.get(url)
            time.sleep(3)
            self.handle_cookies()
            self.scroll_to_load()

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            containers = self.find_product_containers(soup)

            products = []
            for idx, container in enumerate(containers[:max_products], 1):
                try:
                    # Nome prodotto
                    name = self.extract_text_by_class_pattern(container, self.config["name_selectors"])
                    if not name:
                        name = container.get_text(strip=True)[:80]

                    # Prezzo
                    price = self.extract_price(container)

                    # Colori
                    colors_count = self.extract_colors_count(container)

                    # Taglie
                    sizes = self.extract_sizes(container)

                    # Classificazione automatica
                    cat, subcat = classify_product(name, category)

                    # URL prodotto
                    link = container.find('a', href=True)
                    product_url = link['href'] if link else ""
                    if product_url and not product_url.startswith("http"):
                        product_url = self.config["base_url"].rstrip('/') + product_url

                    products.append({
                        "Brand":          self.brand_name,
                        "Modello":        name,
                        "Categoria URL":  category,
                        "Categoria":      cat,
                        "Sottocategoria": subcat,
                        "Prezzo (€)":     price,
                        "N. Colori":      colors_count,
                        "Taglie":         ", ".join(sizes) if sizes else "N/D",
                        "N. Taglie":      len(sizes) if sizes else 0,
                        "URL Prodotto":   product_url,
                        "Data Rilevazione": datetime.now().strftime("%Y-%m-%d %H:%M"),
                    })

                except Exception as e:
                    print(f"      ⚠️  Errore prodotto {idx}: {e}")

            print(f"      ✅ {len(products)} prodotti estratti")
            return products

        except Exception as e:
            print(f"   ❌ Errore categoria {category}: {e}")
            return []

    def scrape_all(self, max_per_category: int = 80) -> List[Dict]:
        """Scrapa tutte le categorie del brand"""
        all_products = []
        for category, url in self.config["category_url_map"].items():
            products = self.scrape_category(category, url, max_per_category)
            all_products.extend(products)
            time.sleep(2)
        return all_products


# ─────────────────────────────────────────────────────────────────────────────
# ANALIZZATORE COMPETITIVO
# ─────────────────────────────────────────────────────────────────────────────

class CompetitiveAnalyzer:
    """
    Orchestratore dell'analisi competitiva multi-brand.
    Avvia il browser, coordina gli scraper, produce il report Excel finale.
    """

    def __init__(self, headless: bool = False, brands: Optional[List[str]] = None):
        self.headless = headless
        self.brands = brands or list(BRAND_CONFIGS.keys())
        self.driver = None
        self.all_data: List[Dict] = []

        print(self._banner())
        self._setup_driver()

    def _banner(self):
        return """
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║      DIADORA COMPETITIVE INTELLIGENCE - MULTI-BRAND ANALYZER v4.0           ║
║                                                                              ║
║      Brand: Diadora | Nike | Adidas | Puma | New Balance | Asics            ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

    def _setup_driver(self):
        options = Options()
        if self.headless:
            options.add_argument('--headless=new')
        options.add_argument('--window-size=1920,1200')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_argument(
            'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        )
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        self.driver.implicitly_wait(8)
        print("✅ WebDriver configurato\n")

    def run(self, max_per_category: int = 80) -> pd.DataFrame:
        """Esegue lo scraping di tutti i brand e restituisce il DataFrame"""
        print("=" * 80)
        print("🚀 AVVIO ANALISI COMPETITIVA")
        print("=" * 80)

        for brand_name in self.brands:
            if brand_name not in BRAND_CONFIGS:
                print(f"⚠️  Brand '{brand_name}' non configurato, skip")
                continue

            print(f"\n{'=' * 60}")
            print(f"  🏷️  BRAND: {brand_name.upper()}")
            print(f"{'=' * 60}")

            config = BRAND_CONFIGS[brand_name]
            scraper = BrandScraper(brand_name, config, self.driver)
            brand_products = scraper.scrape_all(max_per_category)
            self.all_data.extend(brand_products)

            print(f"\n  ✅ {brand_name}: {len(brand_products)} prodotti raccolti")

        df = pd.DataFrame(self.all_data)
        print(f"\n✅ Totale prodotti raccolti: {len(df)}")
        return df

    def build_report(self, df: pd.DataFrame, output_file: str = "Diadora_Competitive_Analysis.xlsx"):
        """
        Costruisce il report Excel con:
          - Sheet 1: Dati grezzi
          - Sheet 2: Analisi per brand e categoria
          - Sheet 3: Analisi prezzi
          - Sheet 4: Analisi varietà (colori / taglie)
        """
        print(f"\n📊 Costruzione report Excel → {output_file}")

        # Pulizia prezzi
        df['Prezzo (€)'] = pd.to_numeric(df['Prezzo (€)'], errors='coerce')

        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

            # ── Sheet 1: Dati grezzi ──────────────────────────────────────
            df.to_excel(writer, sheet_name="Dati Grezzi", index=False)

            # ── Sheet 2: Modelli per brand e categoria ────────────────────
            pivot_models = df.groupby(['Brand', 'Categoria']).size().reset_index(name='N. Modelli')
            pivot_models.to_excel(writer, sheet_name="Modelli per Categoria", index=False)

            # ── Sheet 3: Analisi prezzi ───────────────────────────────────
            price_analysis = df.groupby(['Brand', 'Categoria', 'Sottocategoria'])['Prezzo (€)'].agg(
                Media='mean',
                Minimo='min',
                Massimo='max',
                Mediana='median',
                N_Prodotti='count'
            ).round(2).reset_index()
            price_analysis.to_excel(writer, sheet_name="Analisi Prezzi", index=False)

            # ── Sheet 4: Varietà colori/taglie ───────────────────────────
            variety = df.groupby(['Brand', 'Categoria']).agg(
                Media_Colori=('N. Colori', 'mean'),
                Media_Taglie=('N. Taglie', 'mean'),
                Max_Colori=('N. Colori', 'max'),
                Max_Taglie=('N. Taglie', 'max'),
                N_Modelli=('Modello', 'count')
            ).round(2).reset_index()
            variety.to_excel(writer, sheet_name="Varietà Colori e Taglie", index=False)

            # ── Sheet 5: Pivot prezzo medio per categoria ─────────────────
            pivot_price = df.pivot_table(
                index='Categoria',
                columns='Brand',
                values='Prezzo (€)',
                aggfunc='mean'
            ).round(2)
            pivot_price.to_excel(writer, sheet_name="Pivot Prezzi per Categoria")

        # Formattazione Excel
        self._format_excel(output_file)

        print(f"✅ Report salvato in: {output_file}")
        return output_file

    def _format_excel(self, filename: str):
        """Applica formattazione professionale al report Excel"""
        wb = openpyxl.load_workbook(filename)

        header_fill = PatternFill(start_color="1F3864", end_color="1F3864", fill_type="solid")
        header_font = Font(color="FFFFFF", bold=True, size=11)
        alt_fill    = PatternFill(start_color="DCE6F1", end_color="DCE6F1", fill_type="solid")

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            for cell in ws[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal='center', vertical='center')
            for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
                for cell in row:
                    if row_idx % 2 == 0:
                        cell.fill = alt_fill
            for col in ws.columns:
                max_len = max((len(str(c.value or "")) for c in col), default=10)
                ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

        wb.save(filename)

    def close(self):
        if self.driver:
            self.driver.quit()
            print("✅ Browser chiuso")


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

def main():
    """
    Configurazione principale. Modifica i parametri qui:
      - headless=True  → browser invisibile (più veloce)
      - headless=False → browser visibile (utile per debug)
      - brands=None    → tutti i brand configurati
      - brands=[...]   → solo i brand specificati
    """
    analyzer = CompetitiveAnalyzer(
        headless=False,
        brands=["Diadora", "Nike", "Adidas", "Puma", "New Balance", "Asics"]
    )

    try:
        df = analyzer.run(max_per_category=80)

        if not df.empty:
            analyzer.build_report(df, "Diadora_Competitive_Analysis.xlsx")
            print(f"\n🎉 Analisi completata! {len(df)} prodotti analizzati.")
            print(f"📂 File salvato: Diadora_Competitive_Analysis.xlsx")
        else:
            print("\n⚠️  Nessun dato raccolto. Verifica la connessione e riprova.")

    finally:
        analyzer.close()


if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║      DIADORA COMPETITIVE INTELLIGENCE - MULTI-BRAND ANALYZER v4.0           ║
║                                                                              ║
║      Brand: Diadora | Nike | Adidas | Puma | New Balance | Asics            ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

✅ WebDriver configurato

🚀 AVVIO ANALISI COMPETITIVA

  🏷️  BRAND: DIADORA

   📂 [Diadora] Categoria: Sneakers
      URL: https://www.diadora.com/it/it/calzature/sneakers/
   ⚠️  [Diadora] Nessun cookie banner trovato
   ✅ [Diadora] Trovati 10 prodotti con 'product-tile'
      ✅ 10 prodotti estratti

   📂 [Diadora] Categoria: Running
      URL: https://www.diadora.com/it/it/calzature/running/
   ⚠️  [Diadora] Nessun cookie banner

In [ ]:
#Questa è una di tante query effettuate, per alcuni marchi l'estrapolazione non va a buon fine in quanto il sito dopo diverse prove blocca lo scraping. Tutti idata estrapolati nella prima query sono visibili nel power point.